# C03 — Embeddings e Busca Vetorial

**Disciplina:** Sistemas Cognitivos com Large Language Models (INFNET, 26E2_3)
**Autor:** Anderson Felipe Paixão Corrêa
**Projeto:** Letra da Lei — RAG hierarquia- e vigência-aware sobre o microssistema
penal federal brasileiro

Esta notebook cobre o **ponto 3 da rubrica** (embeddings + busca vetorial). Ela **empacota**
módulos já implementados e testados do pacote `direito_dados` — nenhuma lógica de
recuperação é reimplementada aqui, apenas chamada e narrada.

O que será demonstrado:

1. Carga de um subconjunto do corpus (por velocidade de execução).
2. Chunking *caput*-forward e os prefixos `query:`/`passage:` do e5 multilingue.
3. Indexação vetorial com ChromaDB e consultas de exemplo, com citação.
4. **Segurança de vigência**: filtragem de metadados excluindo dispositivos `REVOGADO`.
5. **Denso vs. híbrido**: comparação entre `VectorIndex.query` e `hybrid_search` (BM25 + denso).
6. **Avaliação de recuperação**: um gold-set pequeno, `hit_rate`/`MRR`, e um caso de falha
   discutido em detalhe.


## Setup

Importamos apenas a API pública dos módulos de corpus e retrieval — a mesma usada pelos
114 testes do projeto.


In [1]:
import sys
import time
from pathlib import Path

# garante que o pacote local seja importável mesmo se o kernel iniciar em outro cwd
_repo_root = Path.cwd()
if not (_repo_root / "direito_dados").exists():
    _repo_root = Path(__file__).resolve().parent if "__file__" in globals() else _repo_root
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from direito_dados.corpus import load_corpus, NORMS
from direito_dados.retrieval.chunks import chunk_corpus
from direito_dados.retrieval.embedder import E5Embedder
from direito_dados.retrieval.index import VectorIndex
from direito_dados.retrieval.lexical import BM25Index, hybrid_search
from direito_dados.retrieval.evaluate import GoldItem, evaluate

print("Diretório de trabalho:", Path.cwd())


Diretório de trabalho: /Users/anderson.correa/andersonfpcorrea/infnet-deliveries/26E2_3


## 1. Subconjunto do corpus

O corpus completo tem 9 normas e 2.310 artigos (ver `c07_lei_como_dado.ipynb` para a
análise sobre o corpus inteiro). Para esta notebook, que gera embeddings reais com o
modelo `intfloat/multilingual-e5-base` (rodando em CPU), usamos um subconjunto **CP
(Código Penal) + L11343 (Lei de Drogas)** — os dois núcleos temáticos mais usados nos
exemplos abaixo (crimes contra a pessoa/patrimônio e tráfico de drogas), suficiente para
demonstrar toda a API de recuperação em segundos, sem sacrificar realismo.


In [2]:
corpus = load_corpus("data/raw", specs=[NORMS["CP"], NORMS["L11343"]])
print(f"Normas carregadas: {[n.id for n in corpus.norms]}")
print(f"Total de artigos: {len(corpus.all_articles())}")
print(f"Em vigor: {len(corpus.in_force_articles())}")


Normas carregadas: ['CP', 'L11343']
Total de artigos: 545
Em vigor: 520


## 2. Chunking *caput*-forward e prefixos e5

`chunk_corpus` gera um `Chunk` por artigo. O campo `embed_text` é o que efetivamente vira
embedding — **não** é o texto bruto do artigo, mas `"{caput}. {texto}"` truncado a 300
caracteres. Isso é uma escolha deliberada de *caput-forward chunking*: o *caput* (a frase
que abre o artigo, normalmente a descrição nuclear da conduta) é repetido no início do
texto embutido, dando a ele peso extra na representação semântica — muitos artigos têm
parágrafos e incisos longos que, sozinhos, diluiriam o sinal do *caput* se fossem
embutidos sem reforço.

O `E5Embedder` por sua vez aplica os prefixos assimétricos exigidos pelo modelo
`multilingual-e5-base`: textos indexados (passagens) recebem o prefixo `"passage: "` e
consultas recebem `"query: "` — é assim que o e5 foi treinado (contrastive learning com
pares query/passage), e omitir os prefixos degrada a qualidade da recuperação.


In [3]:
chunks = chunk_corpus(corpus)
print(f"Chunks gerados (1 por artigo com texto não vazio): {len(chunks)}")

exemplo = next(c for c in chunks if c.id == "CP:art121")
print("\n--- Exemplo: CP:art121 (homicídio) ---")
print("id:        ", exemplo.id)
print("citation:  ", exemplo.metadata["citation"])
print("status:    ", exemplo.metadata["status"])
print("text[:120]:", repr(exemplo.text[:120]))
print("embed_text:", repr(exemplo.embed_text))


Chunks gerados (1 por artigo com texto não vazio): 545

--- Exemplo: CP:art121 (homicídio) ---
id:         CP:art121
citation:   CP art. 121
status:     alterado
text[:120]: 'Matar alguem:\nPena - reclusão, de seis a vinte anos.\nCaso de diminuição de pena\n§\n1º Se o agente comete o crime impelido'
embed_text: 'Matar alguem:. Matar alguem:\nPena - reclusão, de seis a vinte anos.\nCaso de diminuição de pena\n§\n1º Se o agente comete o crime impelido por motivo de relevante valor social ou moral, ou\nsob o domínio de violenta emoção, logo em seguida a injusta provocação da vítima, o\njuiz pode reduzir a pena de um'


### Higiene de dados: ids duplicados no snapshot bruto

O parser deriva o número do artigo a partir do texto consolidado do Planalto. Em L11343,
alguns incisos com sufixo de letra do capítulo "Disposições Finais e Transitórias"
(ex.: Art. 8º-A a 8º-F, um item `(VETADO)`) colidem no mesmo id de chunk
(`L11343:art8`) — um artefato conhecido de parsing, não um bug de recuperação. O
ChromaDB exige ids únicos, então deduplicamos aqui (mantendo a primeira ocorrência) antes
de indexar. Isso é uma etapa de preparação de dados desta notebook, não uma mudança no
pacote `direito_dados`.


In [4]:
seen: set[str] = set()
unique_chunks = []
for c in chunks:
    if c.id in seen:
        continue
    seen.add(c.id)
    unique_chunks.append(c)

print(f"Chunks brutos:  {len(chunks)}")
print(f"Chunks únicos:  {len(unique_chunks)}  (descartados: {len(chunks) - len(unique_chunks)})")


Chunks brutos:  545
Chunks únicos:  524  (descartados: 21)


## 3. Índice vetorial (ChromaDB)

`VectorIndex.build` carrega o modelo e5 (download único na primeira execução), embute
todas as passagens com o prefixo `passage:` e monta uma coleção ChromaDB em memória com
distância de cosseno.


In [5]:
embedder = E5Embedder()  # intfloat/multilingual-e5-base

t0 = time.time()
index = VectorIndex.build(unique_chunks, embedder)
print(f"Índice construído em {time.time() - t0:.1f}s ({len(unique_chunks)} passagens)")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Índice construído em 12.9s (524 passagens)


## 4. Consultas de exemplo

Consultas em linguagem natural, em português — o embedder aplica o prefixo `query:`
internamente. Cada resultado traz `citation` (pronta para exibição), `score` (similaridade
de cosseno) e `metadata` (status de vigência, domínio, nível hierárquico).


In [6]:
def mostra_resultados(query, resultados):
    print(f'Consulta: "{query}"')
    for r in resultados:
        print(f"  {r.citation:<28} score={r.score:.3f}  status={r.metadata['status']}")
    print()

for q in [
    "qual a pena para quem mata alguém?",
    "furto de coisa alheia móvel",
    "tráfico ilícito de entorpecentes",
]:
    mostra_resultados(q, index.query(q, embedder, k=5))


Consulta: "qual a pena para quem mata alguém?"
  CP art. 121                  score=0.891  status=alterado
  CP art. 121-B                score=0.866  status=alterado
  CP art. 226                  score=0.861  status=alterado
  CP art. 147                  score=0.858  status=alterado
  CP art. 121-A                score=0.858  status=alterado

Consulta: "furto de coisa alheia móvel"
  CP art. 168                  score=0.892  status=alterado
  CP art. 155                  score=0.877  status=alterado
  CP art. 157                  score=0.871  status=alterado
  CP art. 169                  score=0.846  status=vigente
  CP art. 312                  score=0.846  status=vigente

Consulta: "tráfico ilícito de entorpecentes"
  L11343 art. 18               score=0.854  status=vigente
  L11343 art. 17               score=0.850  status=alterado
  L11343 art. 50-A             score=0.844  status=alterado
  L11343 art. 22-B             score=0.842  status=alterado
  L11343 art. 28             

## 5. Segurança de vigência: filtragem por metadados

O comportamento **padrão** de `VectorIndex.query` é `exclude_revoked=True` — dispositivos
com `status == "revogado"` nunca aparecem em uma resposta a menos que sejam pedidos
explicitamente. Isso é a forma, em nível de recuperação, da propriedade central do
projeto: **nunca apresentar lei revogada como se fosse vigente**.

Para tornar o efeito visível, usamos duas consultas desenhadas para casar semanticamente
com artigos hoje revogados do Código Penal:

- `"usurpação de nome ou pseudônimo alheio"` → CP art. 185 (revogado pela Lei nº 10.695/2003)
- `"violação sexual mediante fraude"` → CP art. 214 (revogado pela Lei nº 12.015/2009)


In [7]:
for q in ["usurpação de nome ou pseudônimo alheio", "violação sexual mediante fraude"]:
    print("=" * 70)
    print(f'Consulta: "{q}"')
    print("\n-- exclude_revoked=True (padrão) --")
    mostra_resultados(q, index.query(q, embedder, k=5))
    print("-- exclude_revoked=False --")
    mostra_resultados(q, index.query(q, embedder, k=5, exclude_revoked=False))


Consulta: "usurpação de nome ou pseudônimo alheio"

-- exclude_revoked=True (padrão) --
Consulta: "usurpação de nome ou pseudônimo alheio"
  CP art. 242                  score=0.866  status=alterado
  CP art. 162                  score=0.849  status=vigente
  CP art. 307                  score=0.846  status=vigente
  CP art. 313                  score=0.845  status=alterado
  CP art. 168                  score=0.843  status=alterado

-- exclude_revoked=False --
Consulta: "usurpação de nome ou pseudônimo alheio"
  CP art. 242                  score=0.866  status=alterado
  CP art. 162                  score=0.849  status=vigente
  CP art. 307                  score=0.846  status=vigente
  CP art. 313                  score=0.845  status=alterado
  CP art. 168                  score=0.843  status=alterado

Consulta: "violação sexual mediante fraude"

-- exclude_revoked=True (padrão) --
Consulta: "violação sexual mediante fraude"
  CP art. 203                  score=0.849  status=alterado

**Leitura do resultado acima:** o efeito é nítido na segunda consulta: com o filtro
padrão (`exclude_revoked=True`), `CP:art214` (violação sexual mediante fraude, revogado
pela Lei nº 12.015/2009) não aparece entre os 5 primeiros — o índice devolve apenas os
vizinhos semânticos mais próximos *dentre os vigentes*. Com `exclude_revoked=False`, o
mesmo artigo passa a liderar o ranking (maior score de todos, 0.881), pois é de fato o
vizinho semântico mais próximo da consulta — só que hoje sem efeito jurídico algum.

Na primeira consulta (`CP:art185`, usurpação de nome ou pseudônimo alheio, revogado em
2003) o resultado é **idêntico** com o filtro ligado ou desligado — o artigo revogado nem
aparece no top-5 em nenhum dos dois casos. Isso não invalida o mecanismo: o texto de
`CP:art185` no snapshot consolidado é só a nota de revogação em si ("Revogado pela Lei nº
10.695, de 1º.7.2003"), sem nenhum conteúdo semântico sobre usurpação de nome — o *caput*
original foi suprimido do texto consolidado do Planalto, então não há sinal suficiente
para o embedding competir mesmo quando incluído. O contraste entre os dois exemplos é o
ponto: a exclusão de revogados é uma cláusula `where` explícita, aplicada **antes** da
busca por similaridade — ela protege mesmo quando (como no art. 214) o texto revogado
seria, de outra forma, o resultado mais bem ranqueado; quando (como no art. 185) o próprio
texto consolidado já não carrega conteúdo recuperável, o filtro é redundante, mas não
incorreto.

O parâmetro `domain` funciona da mesma forma (filtro exato de metadado) — não exercitado
aqui porque este subconjunto tem um único domínio (`penal`); veja `tests/retrieval/test_index.py`
para o caso multi-domínio.


## 6. Denso vs. híbrido (BM25 + denso)

`BM25Index` é um índice léxico puro (TF-IDF/BM25 clássico, sem dependências externas).
`hybrid_search` funde os dois: normaliza os scores de cada retriever por min-max sobre seu
próprio pool de candidatos e combina como `alpha * denso + (1 - alpha) * léxico`
(`alpha=0.5` por padrão).

Comparamos as três estratégias em duas consultas.


In [8]:
bm25 = BM25Index.build(unique_chunks)

def compara(query, k=5):
    print("=" * 70)
    print(f'Consulta: "{query}"')
    print("  denso :", [r.id for r in index.query(query, embedder, k=k)])
    print("  bm25  :", [r.id for r in bm25.query(query, k=k)])
    print("  híbrido:", [r.id for r in hybrid_search(query, index, bm25, embedder, k=k)])

compara("tráfico ilícito de entorpecentes")
compara("estelionato mediante fraude")


Consulta: "tráfico ilícito de entorpecentes"
  denso : ['L11343:art18', 'L11343:art17', 'L11343:art50-A', 'L11343:art22-B', 'L11343:art28']
  bm25  : ['L11343:art73', 'L11343:art30', 'L11343:art17', 'L11343:art68', 'CP:art83']
  híbrido: ['L11343:art17', 'L11343:art73', 'L11343:art18', 'L11343:art1', 'L11343:art30']
Consulta: "estelionato mediante fraude"
  denso : ['CP:art170', 'CP:art206', 'CP:art183-A', 'CP:art337-L', 'CP:art179']
  bm25  : ['CP:art170', 'CP:art204', 'CP:art171', 'CP:art215', 'CP:art178']


  híbrido: ['CP:art170', 'CP:art204', 'CP:art171', 'CP:art206', 'CP:art183-A']


**Leitura do resultado acima:** na segunda consulta, o CP art. 171 (estelionato) fica de
fora do top-5 **denso** — o nome popular do crime ("Estelionato") é um título de seção no
texto original do Planalto, descartado pelo parser; ele nunca aparece no `caput`/`text` do
artigo, só a redação operativa ("obter vantagem ilícita... mediante artifício, ardil, ou
qualquer outro meio fraudulento"). O BM25, puramente lexical, ainda assim recupera
`CP:art171` (via sobreposição de termos como "mediante" e variações de "fraude"), e o
híbrido herda esse acerto. Este é um exemplo real — não construído — de por que um
retriever só-denso pode falhar quando a consulta usa o nome doutrinário/popular de um crime
que não está literalmente no dispositivo legal, e por que a fusão híbrida é uma rede de
segurança útil. Retomamos esse mesmo caso na avaliação abaixo.


## 7. Avaliação de recuperação (gold-set)

Um gold-set pequeno (6 perguntas) com os ids de dispositivo corretos, avaliado com
`evaluate()` (hit-rate@k e MRR). Cinco perguntas têm resposta dentro do subconjunto
carregado (CP + L11343); a sexta pergunta é **deliberadamente fora do escopo do corpus**
(um artigo do Código de Trânsito Brasileiro, que não faz parte do microssistema penal
federal modelado por este projeto) — incluída para observar como o sistema se comporta
diante de uma pergunta sem resposta indexada.


In [9]:
gold = [
    GoldItem("Qual a pena para quem mata alguém?", ["CP:art121"]),
    GoldItem("Furto de coisa alheia móvel", ["CP:art155"]),
    GoldItem("Roubo mediante grave ameaça ou violência", ["CP:art157"]),
    GoldItem("Porte de drogas para consumo pessoal", ["L11343:art28"]),
    GoldItem("Associação de duas ou mais pessoas para o tráfico", ["L11343:art35"]),
    GoldItem("Homicídio culposo na direção de veículo automotor", ["CTB:art302"]),  # fora do corpus (proposital)
]

def retriever_denso(query, k):
    return index.query(query, embedder, k=k)

metrics = evaluate(retriever_denso, gold, k=5)
print("Métricas (retriever denso, k=5):", metrics)

print()
for item in gold:
    resultados = retriever_denso(item.question, 5)
    ids = [r.id for r in resultados]
    acertou = any(rid in item.relevant_ids for rid in ids)
    marca = "ACERTO" if acertou else "FALHA "
    print(f"[{marca}] {item.question}")
    print(f"         esperado={item.relevant_ids}  top-5={ids}")


Métricas (retriever denso, k=5): {'hit_rate': 0.8333333333333334, 'mrr': 0.625, 'n': 6}

[ACERTO] Qual a pena para quem mata alguém?
         esperado=['CP:art121']  top-5=['CP:art121', 'CP:art121-B', 'CP:art258', 'CP:art226', 'CP:art16']
[ACERTO] Furto de coisa alheia móvel
         esperado=['CP:art155']  top-5=['CP:art168', 'CP:art155', 'CP:art157', 'CP:art312', 'CP:art169']
[ACERTO] Roubo mediante grave ameaça ou violência
         esperado=['CP:art157']  top-5=['CP:art359-L', 'CP:art337-K', 'CP:art359-J', 'CP:art157', 'CP:art359-M']
[ACERTO] Porte de drogas para consumo pessoal
         esperado=['L11343:art28']  top-5=['L11343:art28', 'L11343:art22-B', 'CP:art270', 'L11343:art66', 'L11343:art26']


[ACERTO] Associação de duas ou mais pessoas para o tráfico
         esperado=['L11343:art35']  top-5=['L11343:art35', 'CP:art288', 'L11343:art37', 'L11343:art26-A', 'CP:art149-A']
[FALHA ] Homicídio culposo na direção de veículo automotor
         esperado=['CTB:art302']  top-5=['CP:art73', 'CP:art74', 'CP:art359-M-A', 'L11343:art62', 'L11343:art10']


### Discussão do caso de falha

A pergunta sobre homicídio culposo na direção de veículo — cuja resposta correta
(`CTB:art302`) está no Código de Trânsito Brasileiro — falha, como esperado: o corpus deste
projeto cobre o microssistema penal federal (CF, CP, CPP, LEP e as leis extravagantes
penais listadas em `NORMS`), e o CTB não faz parte dele. Como `VectorIndex.query` sempre
devolve os *k* vizinhos mais próximos (não há limiar de abstenção nesta camada), a consulta
retorna os artigos do CP/L11343 mais próximos semanticamente — sobre culpa, trânsito,
condução — mesmo sabendo que nenhum é a resposta certa. Isso é útil e honesto na avaliação
(mede exatamente o que deveria: 0 nesse item), mas evidencia um limite real do design:
recuperação por *k* fixo não distingue "não sei" de "a melhor opção que tenho". A camada de
geração (`direito_dados.generation.rag`) trata esse problema à parte, com verificação de
citação e abstenção — fora do escopo desta notebook (foco em recuperação), mas é o
próximo elo da cadeia.

O `hit_rate` obtido (5/6 ≈ 0,83) reflete majoritariamente perguntas plenamente respondidas
pelo subconjunto carregado; a única falha é estruturalmente esperada (fora de escopo), não
um erro de embedding.


## Conclusão

Esta notebook demonstrou, sobre módulos já testados do `direito_dados`, o pipeline
completo de recuperação: chunking *caput*-forward → embeddings e5 com prefixos
assimétricos → índice ChromaDB com filtragem de metadados → comparação denso/BM25/híbrido
→ avaliação com gold-set. O ponto mais importante para um sistema de "direito como dado" é
o de segurança de vigência (seção 5): a exclusão de dispositivos revogados por padrão não é
um efeito colateral do ranking semântico, é uma condição de filtro explícita aplicada antes
da consulta vetorial — a garantia central de que este RAG nunca cita, por engano, uma lei
que não vale mais.
